# 2차 — 최종 푸시: 풍부한 피처셋 CatBoost + ECDF 선형회귀 스태킹

## 목표: 실제 제출 0.742 돌파

## 0순위 (이 노트북 돌리기 전에)
**`1차_final_team_blend_submit_FIXED`(OOF 0.74071)를 아직 실제 제출 안 하셨으면 지금 먼저 제출하세요.**
OOF→실제 격차가 보통 +0.0012~0.0015였으니, 이미 0.742 근처일 수 있습니다.

## 이 노트북의 두 가지 새 시도
1. **CatBoost + 풍부한 피처셋** (Keep-NaN/클리핑/안전비율/실효나이/시술유형 토큰화/기증자 회춘 격차) —
   지금까지 LightGBM으로만 테스트했던 피처셋을 CatBoost(NaN 네이티브 처리가 더 강함)에 적용
2. **ECDF 랭크변환 + 선형회귀 메타러너** — 기존 블렌딩(가중치 0~1, 합=1 제약)과 달리
   **음수 가중치 허용** → 모델간 공유 노이즈 상쇄 가능. blend_cache의 모든 기존 후보 +
   이번에 새로 만드는 모델까지 전부 포함해서 스태킹.

## "무한 반복" 가이드
이 노트북 끝에 `run_new_model(name, build_fn)` 패턴을 만들어둘게요. 새 모델 아이디어가 생기면
그 패턴대로 함수만 추가하고 다시 스태킹 셀을 돌리면 됩니다 — 후보가 늘어날수록 ECDF
스태킹이 자동으로 더 많은 조합을 고려해요.

## 누수 방지
카테고리/비율 모두 train 기준만 사용, ECDF는 매 fold의 train 부분으로만 fit.

---
**저장 위치:** `experiment_history/2차/2차_final_push_stacking.ipynb`


In [15]:
import json
import time
import glob
from pathlib import Path

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
import warnings

warnings.filterwarnings("ignore")


In [16]:
DATA_DIR = Path("../../data")
SHARED_DIR = Path("..")
BLEND_DIR = SHARED_DIR / "blend_cache"
SUBMIT_DIR = Path("../submit file")
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
TARGET_COL = "임신 성공 여부"
DEAD_COLS = ["불임 원인 - 여성 요인", "난자 채취 경과일"]
N_SPLITS = 5

EMBRYO_COUNT_COLS = ["총 생성 배아 수","미세주입된 난자 수","미세주입에서 생성된 배아 수","이식된 배아 수",
    "미세주입 배아 이식 수","저장된 배아 수","미세주입 후 저장된 배아 수","해동된 배아 수",
    "해동 난자 수","수집된 신선 난자 수","저장된 신선 난자 수","혼합된 난자 수",
    "파트너 정자와 혼합된 난자 수","기증자 정자와 혼합된 난자 수"]
EMBRYO_BINARY_COLS = ["단일 배아 이식 여부","착상 전 유전 진단 사용 여부","동결 배아 사용 여부",
    "신선 배아 사용 여부","기증 배아 사용 여부","대리모 여부"]
CLIP_RULES = {"총 생성 배아 수":40,"수집된 신선 난자 수":40,"미세주입된 난자 수":45,
    "혼합된 난자 수":40,"저장된 배아 수":30,"배아 이식 경과일":7,"난자 혼합 경과일":7,
    "배아 해동 경과일":2,"난자 해동 경과일":1}
RATIO_SPECS = [
    ("총 생성 배아 수", "혼합된 난자 수", "fertilization_rate"),
    ("미세주입에서 생성된 배아 수", "미세주입된 난자 수", "icsi_fertilization_rate"),
    ("이식된 배아 수", "총 생성 배아 수", "embryo_utilization_rate"),
    ("저장된 배아 수", "총 생성 배아 수", "embryo_freezing_rate"),
    ("혼합된 난자 수", "수집된 신선 난자 수", "oocyte_utilization_rate"),
    ("이식된 배아 수", "해동된 배아 수", "thawed_embryo_transfer_ratio"),
]


## 1. 풍부한 피처셋 빌드 (Keep-NaN + 클리핑 + 비율 + 실효나이 + 시술유형 토큰화)

In [24]:
train_raw = pd.read_csv(TRAIN_PATH)
test_raw_full = pd.read_csv(TEST_PATH)
test_ids = test_raw_full["ID"]
y = train_raw[TARGET_COL].values.astype(int)


def safe_ratio(df, numerator, denominator, new_col):
    df = df.copy()
    can = df[numerator].notna() & df[denominator].notna() & (df[denominator] > 0)
    df[f"{new_col}_available"] = can.astype(int)
    df[new_col] = np.where(can, df[numerator] / df[denominator], np.nan)
    return df


def build_full_features(df):
    df = df.copy()
    df = df.drop(columns=[c for c in DEAD_COLS if c in df.columns])
    df["is_DI"] = (df["시술 유형"] == "DI").astype(int)
    df["froze_embryo"] = df["동결 배아 사용 여부"].fillna(0).astype(int)

    embryo_cols_present = [c for c in (EMBRYO_COUNT_COLS + EMBRYO_BINARY_COLS) if c in df.columns]
    for col in embryo_cols_present:
        df[f"{col}_missing"] = df[col].isna().astype(int)
        is_di_missing = (df["시술 유형"] == "DI") & df[col].isna()
        df[f"{col}_not_applicable_DI"] = is_di_missing.astype(int)

    for col, upper in CLIP_RULES.items():
        if col in df.columns:
            df[f"{col}_high_flag"] = (df[col] > upper).astype(int)
            df[col] = df[col].clip(upper=upper)

    if "배아 이식 경과일" in df.columns:
        df["transfer_day_0_1_flag"] = df["배아 이식 경과일"].isin([0, 1]).astype(int)
        df["transfer_day_3_flag"] = (df["배아 이식 경과일"] == 3).astype(int)
        df["transfer_day_5_or_more_flag"] = (df["배아 이식 경과일"] >= 5).astype(int)

    for num, den, new in RATIO_SPECS:
        if num in df.columns and den in df.columns:
            df = safe_ratio(df, num, den, new)

    # 실효 나이 (기증 난자면 기증자 나이로 교체)
    patient_mid = {"만18-34세":31,"만35-37세":36,"만38-39세":38.5,"만40-42세":41,
                    "만43-44세":43.5,"만45-50세":47.5,"알 수 없음":np.nan}
    donor_mid = {"만20세 이하":20,"만21-25세":23,"만26-30세":28,"만31-35세":33,
                 "만36-40세":38,"만41-45세":43,"알 수 없음":np.nan}
    if "난자 출처" in df.columns and "시술 당시 나이" in df.columns:
        df["patient_age_mid"] = df["시술 당시 나이"].map(patient_mid)
        donor_age_mid = df["난자 기증자 나이"].map(donor_mid) if "난자 기증자 나이" in df.columns else pd.Series(np.nan, index=df.index)
        donor_known = (df["난자 출처"] == "기증 제공") & donor_age_mid.notna()
        df["effective_maternal_age_mid"] = df["patient_age_mid"]
        df.loc[donor_known, "effective_maternal_age_mid"] = donor_age_mid[donor_known]
        df["donor_rejuvenation_gap"] = 0.0
        df.loc[donor_known, "donor_rejuvenation_gap"] = (
            df.loc[donor_known, "patient_age_mid"] - donor_age_mid[donor_known]
        )

    # 시술유형 세부 토큰화
    if "특정 시술 유형" in df.columns:
        s = df["특정 시술 유형"].astype(str)
        for token in ["IVF", "ICSI", "IUI", "ICI", "GIFT", "FER", "Generic DI", "IVI", "BLASTOCYST", "AH"]:
            safe = token.lower().replace(" ", "_")
            df[f"spec_has_{safe}"] = s.str.contains(token, regex=False, na=False).astype(int)

    # 인터랙션 (카테고리 결합)
    if {"시술 당시 나이", "난자 출처"}.issubset(df.columns):
        df["age_oocyte_source"] = df["시술 당시 나이"].astype(str) + "_" + df["난자 출처"].astype(str)

    return df


train_feat = build_full_features(train_raw.drop(columns=["ID"]))
test_feat = build_full_features(test_raw_full.drop(columns=["ID"]))

X_train_full = train_feat.drop(columns=[TARGET_COL])
X_test_full = test_feat.copy()

# ★ 대회 규칙 준수: 카테고리는 train 기준만. 범주형 NaN(원래 결측 + test에만 있는 새 값 둘 다)은
# CatBoost가 못 받으므로 전부 sentinel 문자열로 흡수
# 범주형 처리 (train 기준만) — sentinel로 결측 흡수

SENTINEL = "정보없음"
obj_cols = X_train_full.select_dtypes(include=["object"]).columns.tolist()
for col in obj_cols:
    X_train_full[col] = X_train_full[col].fillna(SENTINEL)
    X_test_full[col] = X_test_full[col].fillna(SENTINEL)
    train_categories = sorted(set(X_train_full[col].astype(str).unique()) | {SENTINEL})
    X_train_full[col] = pd.Categorical(X_train_full[col].astype(str), categories=train_categories)
    X_test_full[col] = pd.Categorical(X_test_full[col].astype(str), categories=train_categories)
    X_test_full[col] = X_test_full[col].fillna(SENTINEL)
X_test_full = X_test_full[X_train_full.columns]

print(f"풍부한 피처셋: train {X_train_full.shape}, test {X_test_full.shape}, 범주형 {len(obj_cols)}개")


풍부한 피처셋: train (256351, 145), test (90067, 145), 범주형 21개


In [22]:
# === 클리핑 제거 비교 (1번 셀 다음에 새 셀로 추가) ===

def build_full_features_noclip(df):
    df = df.copy()
    df = df.drop(columns=[c for c in DEAD_COLS if c in df.columns])
    df["is_DI"] = (df["시술 유형"] == "DI").astype(int)
    df["froze_embryo"] = df["동결 배아 사용 여부"].fillna(0).astype(int)

    embryo_cols_present = [c for c in (EMBRYO_COUNT_COLS + EMBRYO_BINARY_COLS) if c in df.columns]
    for col in embryo_cols_present:
        df[f"{col}_missing"] = df[col].isna().astype(int)
        is_di_missing = (df["시술 유형"] == "DI") & df[col].isna()
        df[f"{col}_not_applicable_DI"] = is_di_missing.astype(int)

    # ↓ 클리핑 블록 전체 생략 (CLIP_RULES 루프 없음) — 이게 유일한 차이

    if "배아 이식 경과일" in df.columns:
        df["transfer_day_0_1_flag"] = df["배아 이식 경과일"].isin([0, 1]).astype(int)
        df["transfer_day_3_flag"] = (df["배아 이식 경과일"] == 3).astype(int)
        df["transfer_day_5_or_more_flag"] = (df["배아 이식 경과일"] >= 5).astype(int)

    for num, den, new in RATIO_SPECS:
        if num in df.columns and den in df.columns:
            df = safe_ratio(df, num, den, new)

    patient_mid = {"만18-34세": 31, "만35-37세": 36, "만38-39세": 38.5, "만40-42세": 41,
                    "만43-44세": 43.5, "만45-50세": 47.5, "알 수 없음": np.nan}
    donor_mid = {"만20세 이하": 20, "만21-25세": 23, "만26-30세": 28, "만31-35세": 33,
                 "만36-40세": 38, "만41-45세": 43, "알 수 없음": np.nan}
    df["patient_age_mid"] = df["시술 당시 나이"].map(patient_mid)
    donor_age_mid = df["난자 기증자 나이"].map(donor_mid)
    donor_known = (df["난자 출처"] == "기증 제공") & donor_age_mid.notna()
    df["effective_maternal_age_mid"] = df["patient_age_mid"]
    df.loc[donor_known, "effective_maternal_age_mid"] = donor_age_mid[donor_known]
    df["donor_rejuvenation_gap"] = 0.0
    df.loc[donor_known, "donor_rejuvenation_gap"] = (
        df.loc[donor_known, "patient_age_mid"] - donor_age_mid[donor_known]
    )

    s = df["특정 시술 유형"].astype(str)
    for token in ["IVF", "ICSI", "IUI", "ICI", "GIFT", "FER", "Generic DI", "IVI", "BLASTOCYST", "AH"]:
        safe = token.lower().replace(" ", "_")
        df[f"spec_has_{safe}"] = s.str.contains(token, regex=False, na=False).astype(int)

    df["age_oocyte_source"] = df["시술 당시 나이"].astype(str) + "_" + df["난자 출처"].astype(str)
    return df


# 클리핑 없는 피처셋 생성 (train만 — 비교는 OOF만 보면 되니 test는 안 만듦)
train_noclip_df = build_full_features_noclip(train_raw.drop(columns=["ID"]))
X_train_noclip = train_noclip_df.drop(columns=[TARGET_COL])

# 범주형 처리 (train 기준만)
SENTINEL = "정보없음"
for col in X_train_noclip.select_dtypes(include=["object"]).columns:
    X_train_noclip[col] = X_train_noclip[col].fillna(SENTINEL)
    cats = sorted(set(X_train_noclip[col].astype(str).unique()) | {SENTINEL})
    X_train_noclip[col] = pd.Categorical(X_train_noclip[col].astype(str), categories=cats)
    
print(f"클리핑 있음 피처수: {X_train_full.shape[1]}, 클리핑 없음 피처수: {X_train_noclip.shape[1]}")


# 단일 시드 5-fold OOF만 빠르게 비교 (test 예측 없이)
def quick_oof_catboost(X, y, seed=42, n_splits=5):
    cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
    cat_idx = [X.columns.get_loc(c) for c in cat_cols]
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof = np.zeros(len(y))
    for tr_idx, val_idx in skf.split(X, y):
        tp = Pool(X.iloc[tr_idx], y[tr_idx], cat_features=cat_idx)
        vp = Pool(X.iloc[val_idx], y[val_idx], cat_features=cat_idx)
        m = CatBoostClassifier(
            loss_function="Logloss", eval_metric="AUC", iterations=1500, learning_rate=0.03,
            depth=6, l2_leaf_reg=5, random_seed=seed, early_stopping_rounds=100,
            allow_writing_files=False, verbose=False,
        )
        m.fit(tp, eval_set=vp, use_best_model=True)
        oof[val_idx] = m.predict_proba(vp)[:, 1]
    return oof


start = time.time()
oof_clip_check = quick_oof_catboost(X_train_full, y)
print(f"클리핑 있음: AUC={roc_auc_score(y, oof_clip_check):.5f}  ({time.time()-start:.0f}s)")

start = time.time()
oof_noclip_check = quick_oof_catboost(X_train_noclip, y)
print(f"클리핑 없음: AUC={roc_auc_score(y, oof_noclip_check):.5f}  ({time.time()-start:.0f}s)")

print(f"\n차이(있음 - 없음): {roc_auc_score(y, oof_clip_check) - roc_auc_score(y, oof_noclip_check):+.5f}")

클리핑 있음 피처수: 145, 클리핑 없음 피처수: 136
클리핑 있음: AUC=0.74047  (815s)
클리핑 없음: AUC=0.74037  (710s)

차이(있음 - 없음): +0.00009


## 2. CatBoost 후보들 (donor_cat 스타일 + alpha_cat 스타일)

In [15]:
def run_catboost(X, y, X_test, seed, class_weights=None, n_splits=N_SPLITS):
    cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
    cat_idx = [X.columns.get_loc(c) for c in cat_cols]
    params = dict(loss_function="Logloss", eval_metric="AUC", iterations=1500, learning_rate=0.03,
                  depth=6, l2_leaf_reg=5, random_seed=seed, early_stopping_rounds=100,
                  allow_writing_files=False, verbose=False)
    if class_weights is not None:
        params["class_weights"] = class_weights

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof = np.zeros(len(y))
    test_pred = np.zeros(len(X_test))
    for tr_idx, val_idx in skf.split(X, y):
        tp = Pool(X.iloc[tr_idx], y[tr_idx], cat_features=cat_idx)
        vp = Pool(X.iloc[val_idx], y[val_idx], cat_features=cat_idx)
        m = CatBoostClassifier(**params)
        m.fit(tp, eval_set=vp, use_best_model=True)
        oof[val_idx] = m.predict_proba(vp)[:, 1]
        test_pred += m.predict_proba(Pool(X_test, cat_features=cat_idx))[:, 1] / n_splits
    return oof, test_pred


# donor_cat 스타일: 7시드 평균
print("=== donor_cat (7시드 평균) ===")
donor_oofs, donor_tests = [], []
start = time.time()
for seed in [42, 2024, 777, 1234, 5678, 9012, 3456]:
    oof_s, test_s = run_catboost(X_train_full, y, X_test_full, seed=seed)
    donor_oofs.append(oof_s); donor_tests.append(test_s)
    print(f"  seed={seed}: AUC={roc_auc_score(y, oof_s):.5f}  ({time.time()-start:.0f}s 누적)")
oof_donor_cat = np.mean(donor_oofs, axis=0)
test_donor_cat = np.mean(donor_tests, axis=0)
print(f"donor_cat(7시드 평균) OOF AUC: {roc_auc_score(y, oof_donor_cat):.5f}")

np.save(BLEND_DIR / "oof_donor_cat.npy", oof_donor_cat)
np.save(BLEND_DIR / "test_donor_cat.npy", test_donor_cat)


=== donor_cat (7시드 평균) ===
  seed=42: AUC=0.74047  (824s 누적)
  seed=2024: AUC=0.74048  (1559s 누적)
  seed=777: AUC=0.74047  (2363s 누적)
  seed=1234: AUC=0.74039  (3094s 누적)
  seed=5678: AUC=0.74042  (3935s 누적)
  seed=9012: AUC=0.74031  (4687s 누적)
  seed=3456: AUC=0.74049  (5528s 누적)
donor_cat(7시드 평균) OOF AUC: 0.74076


In [13]:
# alpha_cat 스타일: class_weights로 약하게 재가중 (단독은 손해 예상, 스태킹 재료용)
print("=== alpha_cat (class_weights=[0.6, 0.4]) ===")
start = time.time()
oof_alpha, test_alpha = run_catboost(X_train_full, y, X_test_full, seed=42, class_weights=[0.60, 0.40])
print(f"alpha_cat OOF AUC: {roc_auc_score(y, oof_alpha):.5f}  ({time.time()-start:.0f}s)")

np.save(BLEND_DIR / "oof_alpha_cat.npy", oof_alpha)
np.save(BLEND_DIR / "test_alpha_cat.npy", test_alpha)


=== alpha_cat (class_weights=[0.6, 0.4]) ===
alpha_cat OOF AUC: 0.74033  (844s)


## 3. LightGBM + 풍부한 피처셋 + scale_pos_weight=2.0 (스태킹 재료용)

In [14]:
# 풍부한 피처셋은 category dtype이라 LightGBM도 네이티브로 바로 사용 가능
print("=== lgbm_spw2_richfeat ===")
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
oof_spw = np.zeros(len(y))
test_spw = np.zeros(len(X_test_full))
params_spw = dict(objective="binary", n_estimators=3000, learning_rate=0.018, num_leaves=63,
                   min_child_samples=60, subsample=0.85, colsample_bytree=0.85,
                   reg_alpha=0.05, reg_lambda=2.0, scale_pos_weight=2.0,
                   random_state=42, verbose=-1, n_jobs=-1)

start = time.time()
for tr_idx, val_idx in skf.split(X_train_full, y):
    m = LGBMClassifier(**params_spw)
    m.fit(X_train_full.iloc[tr_idx], y[tr_idx])
    oof_spw[val_idx] = m.predict_proba(X_train_full.iloc[val_idx])[:, 1]
    test_spw += m.predict_proba(X_test_full)[:, 1] / N_SPLITS
print(f"lgbm_spw2_richfeat OOF AUC: {roc_auc_score(y, oof_spw):.5f}  ({time.time()-start:.0f}s)")

np.save(BLEND_DIR / "oof_lgbm_spw2_richfeat.npy", oof_spw)
np.save(BLEND_DIR / "test_lgbm_spw2_richfeat.npy", test_spw)


=== lgbm_spw2_richfeat ===
lgbm_spw2_richfeat OOF AUC: 0.73297  (192s)


## 4. 기존 blend_cache 전체 자동 스캔 (새 후보 포함)

In [25]:
MANUAL_TEST_MAP = {"oof_10seed_bagged.npy": "test_lgbm_bagged.npy"}
V5_DIR = SHARED_DIR / "팀원파일" / "김영혜"
V5_OOF_PATH = V5_DIR / "v5_ensemble_oof.npy"
V5_SUBMISSION_PATH = V5_DIR / "submission_v5_imputed.csv"

all_npy = sorted(glob.glob(str(BLEND_DIR / "*.npy")))
oof_candidates = [f for f in all_npy if "oof" in Path(f).stem.lower()]

models = {}
for f in oof_candidates:
    fname = Path(f).name
    stem = Path(f).stem
    oof_arr = np.load(f)
    if len(oof_arr) != len(y):
        continue
    if stem.startswith("oof_"):
        name = stem[len("oof_"):]
        default_test = BLEND_DIR / f"test_{name}.npy"
    elif stem.endswith("_oof"):
        name = stem[: -len("_oof")]
        default_test = BLEND_DIR / f"{name}_test.npy"
    else:
        name = stem
        default_test = None
    test_f = BLEND_DIR / MANUAL_TEST_MAP[fname] if fname in MANUAL_TEST_MAP else default_test
    entry = {"oof": oof_arr}
    if test_f is not None and test_f.exists():
        entry["test"] = np.load(test_f)
    models[name] = entry

if V5_OOF_PATH.exists():
    v5_oof = np.load(V5_OOF_PATH)
    if len(v5_oof) == len(y):
        entry = {"oof": v5_oof}
        if V5_SUBMISSION_PATH.exists():
            entry["test"] = pd.read_csv(V5_SUBMISSION_PATH)["probability"].values
        models["v5_ensemble"] = entry

# 안전장치
for n in list(models.keys()):
    if n == "y" or roc_auc_score(y, models[n]["oof"]) >= 0.995:
        del models[n]

model_names_with_test = [n for n in models if "test" in models[n]]
print(f"test 예측 있는 전체 후보 수: {len(model_names_with_test)}")
print(f"{'후보':<28} | {'OOF AUC':>8}")
print("-" * 42)
for n in sorted(model_names_with_test, key=lambda x: -roc_auc_score(y, models[x]["oof"])):
    print(f"{n:<28} | {roc_auc_score(y, models[n]['oof']):>8.5f}")


test 예측 있는 전체 후보 수: 12
후보                           |  OOF AUC
------------------------------------------
donor_cat                    |  0.74076
v5_ensemble                  |  0.74043
10seed_bagged                |  0.74036
alpha_cat                    |  0.74033
xgboost_bagged               |  0.74010
catboost_bagged              |  0.74005
feature_subspace             |  0.73973
xgb_rankpairwise             |  0.73939
lgbm_lambdarank              |  0.73748
lgbm_keepnan                 |  0.73648
mlp                          |  0.73313
lgbm_spw2_richfeat           |  0.73297


## 5. ECDF 랭크변환 + 선형회귀 메타러너 (음수 가중치 허용, fold-safe)

In [26]:
class ECDFReference:
    """fold-train 분포로 fit. transform은 그 분포 기준 평균순위/n (동점 보정)."""
    def __init__(self, ref):
        self.sorted = np.sort(np.asarray(ref, dtype=float))
        self.n = max(len(self.sorted), 1)

    def transform(self, x):
        x = np.asarray(x, dtype=float)
        left = np.searchsorted(self.sorted, x, side="left")
        right = np.searchsorted(self.sorted, x, side="right")
        return (left + right) / 2.0 / self.n


def make_ecdf_linear_stack(names, models, y, n_splits=N_SPLITS, seed=42):
    oof_matrix = np.column_stack([models[n]["oof"] for n in names])
    has_test = all("test" in models[n] for n in names)
    test_matrix = np.column_stack([models[n]["test"] for n in names]) if has_test else None

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof_pred = np.zeros(len(y))
    test_pred = np.zeros(test_matrix.shape[0]) if has_test else None
    ncol = len(names)

    for tr, va in skf.split(oof_matrix, y):
        Xtr = np.empty((len(tr), ncol))
        Xva = np.empty((len(va), ncol))
        Xte = np.empty((test_matrix.shape[0], ncol)) if has_test else None
        for c in range(ncol):
            ref = ECDFReference(oof_matrix[tr, c])
            Xtr[:, c] = ref.transform(oof_matrix[tr, c])
            Xva[:, c] = ref.transform(oof_matrix[va, c])
            if has_test:
                Xte[:, c] = ref.transform(test_matrix[:, c])
        model = make_pipeline(StandardScaler(), LinearRegression())
        model.fit(Xtr, y[tr])
        oof_pred[va] = np.clip(model.predict(Xva), 0, 1)
        if has_test:
            test_pred += np.clip(model.predict(Xte), 0, 1) / n_splits

    auc = roc_auc_score(y, oof_pred)
    return oof_pred, test_pred, auc


# 여러 조합 시도 (1등 노트북처럼 subset 여러 개 비교)
all_names = model_names_with_test
print(f"전체 {len(all_names)}개 후보로 스태킹 시도\n")

stack_full_oof, stack_full_test, stack_full_auc = make_ecdf_linear_stack(all_names, models, y)
print(f"[전체 {len(all_names)}개] ECDF-LinearRegression 스택 OOF AUC: {stack_full_auc:.5f}")

# 상위 N개만 사용한 subset도 비교 (전체가 과적합 위험 있을 수 있음)
top_sorted = sorted(all_names, key=lambda x: -roc_auc_score(y, models[x]["oof"]))
for k in [3, 5, 7]:
    if k <= len(top_sorted):
        subset = top_sorted[:k]
        _, _, auc_k = make_ecdf_linear_stack(subset, models, y)
        print(f"[상위 {k}개: {subset}] OOF AUC: {auc_k:.5f}")

print()
print("비교 기준:")
print("  팀 블렌딩(2개 모델, 가중평균): 0.74071")
print("  1등 전체 아키텍처(7모델+5스택+3단계): 0.740973")


전체 12개 후보로 스태킹 시도

[전체 12개] ECDF-LinearRegression 스택 OOF AUC: 0.74098
[상위 3개: ['donor_cat', 'v5_ensemble', '10seed_bagged']] OOF AUC: 0.74091
[상위 5개: ['donor_cat', 'v5_ensemble', '10seed_bagged', 'alpha_cat', 'xgboost_bagged']] OOF AUC: 0.74091
[상위 7개: ['donor_cat', 'v5_ensemble', '10seed_bagged', 'alpha_cat', 'xgboost_bagged', 'catboost_bagged', 'feature_subspace']] OOF AUC: 0.74096

비교 기준:
  팀 블렌딩(2개 모델, 가중평균): 0.74071
  1등 전체 아키텍처(7모델+5스택+3단계): 0.740973


In [27]:
# =============================================================================
# Ridge 메타러너로 교체 + alpha 그리드 비교
# 놓을 위치: 2차_final_push_stacking.ipynb, "5. ECDF 랭크변환 + 선형회귀 메타러너" 셀 다음
# (ECDFReference, all_names, models, y, N_SPLITS, StandardScaler, make_pipeline,
#  roc_auc_score, np, StratifiedKFold는 이미 노트북에 정의되어 있음)
# =============================================================================
from sklearn.linear_model import Ridge, LinearRegression


def make_ecdf_ridge_stack(names, models, y, alpha=1.0, n_splits=N_SPLITS, seed=42):
    """LinearRegression 대신 Ridge(alpha)를 메타러너로 사용. alpha=0이면 LinearRegression과 동일."""
    oof_matrix = np.column_stack([models[n]["oof"] for n in names])
    has_test = all("test" in models[n] for n in names)
    test_matrix = np.column_stack([models[n]["test"] for n in names]) if has_test else None

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof_pred = np.zeros(len(y))
    test_pred = np.zeros(test_matrix.shape[0]) if has_test else None
    ncol = len(names)

    for tr, va in skf.split(oof_matrix, y):
        Xtr = np.empty((len(tr), ncol))
        Xva = np.empty((len(va), ncol))
        Xte = np.empty((test_matrix.shape[0], ncol)) if has_test else None
        for c in range(ncol):
            ref = ECDFReference(oof_matrix[tr, c])
            Xtr[:, c] = ref.transform(oof_matrix[tr, c])
            Xva[:, c] = ref.transform(oof_matrix[va, c])
            if has_test:
                Xte[:, c] = ref.transform(test_matrix[:, c])

        model = make_pipeline(StandardScaler(), Ridge(alpha=alpha) if alpha > 0 else LinearRegression())
        model.fit(Xtr, y[tr])
        oof_pred[va] = np.clip(model.predict(Xva), 0, 1)
        if has_test:
            test_pred += np.clip(model.predict(Xte), 0, 1) / n_splits

    auc = roc_auc_score(y, oof_pred)
    return oof_pred, test_pred, auc


# --- alpha 그리드 비교 (전체 12개 후보 기준) ---
print(f"{'alpha':>8} | {'OOF AUC':>8}")
print("-" * 21)
ridge_results = {}
for alpha in [0.0, 0.1, 0.3, 0.5, 1.0, 3.0, 10.0, 30.0]:
    _, _, auc = make_ecdf_ridge_stack(all_names, models, y, alpha=alpha)
    ridge_results[alpha] = auc
    label = "0.0 (=LinearRegression)" if alpha == 0.0 else str(alpha)
    print(f"{label:>8} | {auc:.5f}")

best_alpha = max(ridge_results, key=ridge_results.get)
print(f"\n최적 alpha={best_alpha}, OOF AUC={ridge_results[best_alpha]:.5f}")
print(f"참고: LinearRegression(alpha=0) 기준 = {ridge_results[0.0]:.5f}")

# --- 최적 alpha로 최종 test 예측 생성 (기존 6번 셀 대체) ---
FINAL_NAMES_RIDGE = all_names  # 필요하면 top_sorted[:N] 등으로 교체
final_oof_r, final_test_r, final_auc_r = make_ecdf_ridge_stack(
    FINAL_NAMES_RIDGE, models, y, alpha=best_alpha
)
print(f"\n최종(Ridge alpha={best_alpha}) OOF AUC: {final_auc_r:.5f}")

BLEND_DIR.mkdir(exist_ok=True)
np.save(BLEND_DIR / "ecdf_ridge_stack_test.npy", final_test_r)

SUBMIT_DIR.mkdir(exist_ok=True)
submission_r = pd.DataFrame({"ID": test_ids, "probability": final_test_r})
out_path_r = SUBMIT_DIR / f"2차_ecdf_ridge_stack_local{final_auc_r:.5f}.csv"
submission_r.to_csv(out_path_r, index=False)
print(f"저장 완료: {out_path_r}")

   alpha |  OOF AUC
---------------------
0.0 (=LinearRegression) | 0.74098
     0.1 | 0.74098
     0.3 | 0.74098
     0.5 | 0.74098
     1.0 | 0.74098
     3.0 | 0.74098
    10.0 | 0.74098
    30.0 | 0.74097

최적 alpha=0.0, OOF AUC=0.74098
참고: LinearRegression(alpha=0) 기준 = 0.74098

최종(Ridge alpha=0.0) OOF AUC: 0.74098
저장 완료: ../submit file/2차_ecdf_ridge_stack_local0.74098.csv


In [18]:
# === 5-1. Greedy + ECDF선형회귀 선택 (5번 셀 다음에 새 셀로 추가) ===
def greedy_ecdf_selection(pool_names, models, y, improve_threshold=0.00015):
    individual_aucs = {n: roc_auc_score(y, models[n]["oof"]) for n in pool_names}
    selected = [max(individual_aucs, key=individual_aucs.get)]
    remaining = [n for n in pool_names if n != selected[0]]
    current_auc = individual_aucs[selected[0]]
    print(f"시작: {selected[0]} (단독 AUC={current_auc:.5f})\n")

    history = [(list(selected), current_auc)]
    while remaining:
        best_candidate, best_new_auc = None, current_auc
        for cand in remaining:
            _, _, trial_auc = make_ecdf_linear_stack(selected + [cand], models, y)
            if trial_auc > best_new_auc:
                best_new_auc, best_candidate = trial_auc, cand
        improvement = best_new_auc - current_auc
        if best_candidate is None or improvement < improve_threshold:
            print(f"중단 (최선 개선폭 {improvement:+.5f} < {improve_threshold})")
            break
        selected.append(best_candidate)
        remaining.remove(best_candidate)
        current_auc = best_new_auc
        history.append((list(selected), current_auc))
        print(f"+ {best_candidate:<28} -> {current_auc:.5f}  (개선 {improvement:+.5f})")

    return selected, current_auc, history


greedy_selected, greedy_auc, greedy_history = greedy_ecdf_selection(all_names, models, y)
print(f"\n최종 선택: {greedy_selected}")
print(f"Greedy+ECDF 최종 OOF AUC: {greedy_auc:.5f}")
print(f"(참고: 전체 12개 한꺼번에 선형회귀 = 0.74095)")


시작: donor_cat (단독 AUC=0.74076)

중단 (최선 개선폭 +0.00014 < 0.00015)

최종 선택: ['donor_cat']
Greedy+ECDF 최종 OOF AUC: 0.74076
(참고: 전체 12개 한꺼번에 선형회귀 = 0.74095)


## 6. 최선의 스택으로 최종 제출 파일 생성

In [19]:
# 위 결과 중 가장 높은 AUC를 보인 조합으로 교체해서 실행하세요
FINAL_NAMES = all_names  # 또는 top_sorted[:5] 등 위에서 가장 좋았던 조합으로 교체

final_oof, final_test, final_auc = make_ecdf_linear_stack(FINAL_NAMES, models, y)
print(f"최종 선택 조합: {FINAL_NAMES}")
print(f"최종 OOF AUC: {final_auc:.5f}")

BLEND_DIR.mkdir(exist_ok=True)
np.save(BLEND_DIR / "ecdf_stack_test.npy", final_test)

SUBMIT_DIR.mkdir(exist_ok=True)
submission = pd.DataFrame({"ID": test_ids, "probability": final_test})
out_path = SUBMIT_DIR / f"2차_ecdf_stack_local{final_auc:.5f}.csv"
submission.to_csv(out_path, index=False)
print(f"저장 완료: {out_path}")


최종 선택 조합: ['lgbm_lambdarank', '10seed_bagged', 'alpha_cat', 'catboost_bagged', 'donor_cat', 'feature_subspace', 'lgbm_keepnan', 'lgbm_spw2_richfeat', 'mlp', 'xgboost_bagged', 'xgb_rankpairwise', 'v5_ensemble']
최종 OOF AUC: 0.74098
저장 완료: ../submit file/2차_ecdf_stack_local0.74098.csv


## 7. "무한 반복" — 새 모델 추가하는 법

새 모델 아이디어가 생기면:
1. OOF/test 배열 생성 (위 패턴처럼 5-fold CV)
2. `np.save(BLEND_DIR / "oof_{이름}.npy", oof)`, `np.save(BLEND_DIR / "test_{이름}.npy", test)`로 저장
3. 4번 셀(자동 스캔)부터 다시 실행 — 새 모델이 자동으로 풀에 잡힘
4. 5번 셀(ECDF 스태킹)을 다시 돌려서 조합에 도움 되는지 확인

후보가 많아질수록 5번 셀의 "상위 N개" 비교도 늘려보세요(top_sorted[:10] 등).
